# Attention

Attention is not just a part of the transformer. Attention IS the
transformer.

Consider this sentence. The cat sat on the mat because it was warm.
What does it refer to. A human knows it means the mat. But how
would a computer figure this out.

Before attention existed a model would read words one by one left
to right. By the time it reached the word it the word mat was far
in the past. Its information had faded. The model would forget.

Attention solves this by letting every word look at every other
word at the same time. When the model reaches the word it it can
look back at all the previous words and decide which one matters
most. In this case it would look at mat because warmth is a
property of objects not animals.

The way attention decides how much to care about each word is
through something called queries keys and values. Think of it
like being at a party. Your query is what you are looking for.
Every other person has a key that says what they offer. A high
match means you pay attention to that person.

In this notebook we build the full attention mechanism from
scratch. We trace a worked example with real numbers so you
can see exactly what happens inside every modern language model.

## The scale of this notebook

We are building attention at the same size that GPT-2 Small used.
That means d_model of 768 and num_heads of 12. Each head gets 64
dimensions to work with. The full attention layer has about 2.3
million parameters.

This is small enough to run on a laptop but big enough to show
you how real language models work. The math does not change at
bigger sizes. Only the numbers get larger.

Here is what changes when you scale up. The architecture stays
the same. The dimensions just grow. These numbers come from the
original GPT-3 paper published by OpenAI.

| Model | Layers | d_model | Heads | Params |
|---|---|---|---|---|
| GPT-2 Small | 12 | 768 | 12 | 124M |
| GPT-2 Medium | 24 | 1024 | 16 | 350M |
| GPT-2 Large | 36 | 1280 | 20 | 774M |
| GPT-3 350M | 24 | 1024 | 16 | 350M |
| GPT-3 1.3B | 24 | 2048 | 24 | 1.3B |
| GPT-3 6.7B | 32 | 4096 | 32 | 6.7B |
| GPT-3 175B | 96 | 12288 | 96 | 175B |
| LLaMA 7B | 32 | 4096 | 32 | 7B |
| LLaMA 13B | 40 | 5120 | 40 | 13B |
| LLaMA 70B | 80 | 8192 | 64 | 70B |

GPT-3 175B uses the exact same attention formula we are about
to write. It just runs it with 96 layers instead of 12 and with
12288 dimensions instead of 768. The code is identical. The
numbers are just a thousand times bigger.

You cannot run GPT-3 175B on a consumer GPU. It needs several
data center GPUs and millions of dollars in electricity. But the
notebook you are about to run captures the same idea at a size
you can actually play with and understand.

## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Rotary Position Embeddings

We need RoPE from the previous chapter. It rotates query and key
vectors so the attention score depends on relative position.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, d_model, max_seq_len=2048, theta=10000.0):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even"
        dim_indices = torch.arange(0, d_model, 2).float()
        inv_freq = 1.0 / (theta ** (dim_indices / d_model))
        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    @staticmethod
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat([-x2, x1], dim=-1)

    def forward(self, x, seq_len):
        cos = self.cos_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cached[:seq_len].unsqueeze(0).unsqueeze(0)
        return (x * cos) + (self.rotate_half(x) * sin)

## Causal Mask

During training each token can only see tokens before it. This
prevents the model from cheating by looking at future words.

In [ ]:
def create_causal_mask(seq_len, device):
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.view(1, 1, seq_len, seq_len)

## Multi-Head Attention

This is the complete attention module. It does eight things in
order. Project to Q K and V. Reshape into multiple heads. Apply
RoPE. Compute scaled dot product scores. Apply the causal mask.
Run softmax. Weight the values. Merge heads and project out.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rotary = RotaryPositionalEmbedding(self.head_dim)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        qkv = self.qkv_proj(x)
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        q = self.rotary(q, seq_len)
        k = self.rotary(k, seq_len)

        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(attn_scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        attn_output = attn_weights @ v

        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(batch_size, seq_len, self.d_model)

        output = self.out_proj(attn_output)
        output = self.resid_dropout(output)
        return output

## Test the attention module

Create a tiny model with 12 attention heads and feed it a batch
of 8 sequences of 64 tokens each.

In [ ]:
attn = MultiHeadAttention(d_model=768, num_heads=12)

batch_size = 8
seq_len = 64
x = torch.randn(batch_size, seq_len, 768)
mask = create_causal_mask(seq_len, x.device)

output = attn(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Input and output shapes match: {x.shape == output.shape}")
print()

params = sum(p.numel() for p in attn.parameters())
print(f"Attention parameters: {params:,}")

## Worked example with real numbers

Now let us trace through attention with a tiny 3 token sentence.
We will use d_model of 4 and a single head so we can see every
number.

In [ ]:
tiny_attn = MultiHeadAttention(d_model=4, num_heads=1)

tokens = torch.tensor([[
    [0.5,  0.2, -0.3,  0.8],
    [0.1, -0.5,  0.7, -0.2],
    [0.9,  0.3, -0.1, -0.5],
]])
print(f"Input tokens shape: {tokens.shape}")
print()

with torch.no_grad():
    qkv = tiny_attn.qkv_proj(tokens)
    qkv = qkv.reshape(1, 3, 3, 1, 4)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv[0], qkv[1], qkv[2]

print("Query vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in q[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

print("Key vectors:")
for i in range(3):
    vals = [f"{v:.3f}" for v in k[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")
print()

scores = (q @ k.transpose(-2, -1)) / math.sqrt(4)
print("Attention scores (before mask and softmax):")
print(f"  Token 0: {[f'{v:.3f}' for v in scores[0, 0, 0].tolist()]}")
print(f"  Token 1: {[f'{v:.3f}' for v in scores[0, 0, 1].tolist()]}")
print(f"  Token 2: {[f'{v:.3f}' for v in scores[0, 0, 2].tolist()]}")

## Apply causal mask and softmax

The causal mask sets future positions to negative infinity.
After softmax those positions become zero. Each token can only
attend to itself and tokens that came before it.

In [ ]:
mask = create_causal_mask(3, scores.device)
scores_masked = scores.masked_fill(mask == 0, float('-inf'))
weights = F.softmax(scores_masked, dim=-1)

print("Attention weights (after mask and softmax):")
for i in range(3):
    row = [f"{v:.3f}" for v in weights[0, 0, i].tolist()]
    print(f"  Token {i} attends to: [{', '.join(row)}]")
print()
print("Token 0 sees only itself. Token 1 sees 0 and 1. Token 2 sees all three.")
print()

output = weights @ v
print("Output vectors (weighted sum of values):")
for i in range(3):
    vals = [f"{v:.3f}" for v in output[0, 0, i].tolist()]
    print(f"  Token {i}: [{', '.join(vals)}]")

## Visualizing the causal attention matrix

Each row is a token. Each column is what it attends to. The
upper triangle is empty because future tokens are blocked.

In [ ]:
print("Causal attention matrix (Token i attending to Token j):")
print("          T0      T1      T2")
for i in range(3):
    row = "  ".join([f"{weights[0, 0, i, j]:.3f}" for j in range(3)])
    print(f"T{i}       {row}")
print()
print("The upper right is zero. The model cannot see the future.")